In [1]:
# Parameters
BATCH_MODE = True


# Guessing Game: Father Fouras vs Laurent Jalabert

**Navigation**: [Index](../README.md)

In this notebook, we will simulate the legendary duel between Father Fouras and Laurent Jalabert using Semantic Kernel with conversational agents.

In [2]:
# Import guards - availability flags for external dependencies

try:
    from dotenv import load_dotenv
    DOTENV_AVAILABLE = True
except ImportError:
    DOTENV_AVAILABLE = False
    print(f'  dotenv non disponible - certaines fonctionnalites seront limitees')

try:
    import semantic_kernel
    SEMANTIC_KERNEL_AVAILABLE = True
except ImportError:
    SEMANTIC_KERNEL_AVAILABLE = False
    print(f'  semantic_kernel non disponible - certaines fonctionnalites seront limitees')

# Bloc 1 - Installation et imports

import os
import logging
from dotenv import load_dotenv
from semantic_kernel import Kernel
from semantic_kernel.agents import ChatCompletionAgent, AgentGroupChat
from semantic_kernel.agents.strategies import KernelFunctionTerminationStrategy
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import ChatHistory
from semantic_kernel.functions import KernelArguments

# Configuration des logs
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger('FortBoyard')

# Chargement des variables d'environnement
env_path = os.path.join(os.path.dirname(os.path.abspath(".")), ".env")
if os.path.exists(env_path):
    load_dotenv(env_path)
else:
    load_dotenv()


## Agent configuration

In [3]:
# Bloc 2 - Création du kernel
MOT_A_DEVINER = "anticonstitutionnellement"

def create_kernel():
    kernel = Kernel()
    model_id = os.getenv("OPENAI_CHAT_MODEL_ID", "gpt-5-mini")
    kernel.add_service(OpenAIChatCompletion(
        service_id="openai",
        ai_model_id=model_id,
        api_key=os.getenv("OPENAI_API_KEY")
    ))
    return kernel


## Definition of prompts

In [4]:
# Bloc 3 - Prompts des agents
PERE_FOURAS_PROMPT = f"""
Tu es le Père Fouras de Fort Boyard. 
Tu dois faire deviner le mot '{MOT_A_DEVINER}'. 
Utilise des charades et réponses énigmatiques. 
Ne révèle jamais directement le mot !
"""

LAURENT_JALABERT_PROMPT = """
Tu es Laurent Jalabert. 
Tu dois deviner le mot en posant des questions fermées (Oui/Non).
Sois perspicace et stratégique dans tes questions.
"""

### Exercise 1: Customize the duel characters

The current prompts define Père Fouras as enigmatic and Laurent Jalabert as perceptive. The goal is to create a variant of the game with two different characters: a **Detective** (who asks deductive questions) and a **Witness** (who answers evasively without revealing the word).

**Objective**: write the system instructions for these two new characters and configure the corresponding agents.

**Hints**:
- # Step 1: Write the Witness prompt (role, rules, prohibitions)
- # Step 2: Write the Detective prompt (questioning strategy)
- # Hint: the Witness must never reveal the word directly, but may give indirect clues

In [1]:
# Exercice 1 : Personnaliser les personnages du duel
# TODO etudiant : definir les prompts et creer les agents

TEMoin_PROMPT = ""  # Etape 1 : instructions du Temoin (evasif, indices indirects)
DETECTIVE_PROMPT = ""  # Etape 2 : instructions du Detective (questions deductives)

# Indice : creer les agents avec ChatCompletionAgent(kernel=..., name=..., instructions=...)
temoin = None    # TODO etudiant : creer l'agent Temoin
detective = None  # TODO etudiant : creer l'agent Detective

print("Exercice a completer : personnages Detective et Temoin")

Exercice a completer


## Creating agents with custom strategies

In [5]:
# Bloc 4 - Définition des agents
pere_fouras = ChatCompletionAgent(
    kernel=create_kernel(),
    name="Pere_Fouras",
    instructions=PERE_FOURAS_PROMPT,
)

laurent_jalabert = ChatCompletionAgent(
    kernel=create_kernel(),
    name="Laurent_Jalabert",
    instructions=LAURENT_JALABERT_PROMPT,
)

## Custom termination strategy

In [6]:
from semantic_kernel.agents.strategies.termination.termination_strategy import TerminationStrategy
from semantic_kernel.contents.chat_message_content import ChatMessageContent

# Bloc 5 - Logique de terminaison
class FortBoyardTerminationStrategy(TerminationStrategy):
    """Arrête la partie si le mot est deviné"""
    
    async def should_agent_terminate(
        self, 
        agent: ChatCompletionAgent, 
        history: list[ChatMessageContent], 
        cancellation_token = None
    ) -> bool:
        if not history:
            return False
        
        last_message = str(history[-1].content).lower()
        return MOT_A_DEVINER in last_message

### Exercise 2: Termination with giving-up detection

The current strategy stops the game only if the word is guessed. But Laurent Jalabert could also give up. The goal is to create a hybrid strategy that stops the game if the word is guessed **or** if Laurent expresses giving up ("je donne ma langue au chat", "j'abandonne").

**Objective**: implement `RenoncementTerminationStrategy` which combines detection of the secret word with detection of giving-up phrases.

**Hints**:
- # Step 1: Define a list of typical giving-up phrases
- # Step 2: Check whether the last message contains one of these phrases (in lowercase)
- # Hint: combine the two conditions with `or` in `should_agent_terminate`

In [1]:
from semantic_kernel.agents.strategies.termination.termination_strategy import TerminationStrategy
from semantic_kernel.contents.chat_message_content import ChatMessageContent

class RenoncementTerminationStrategy(TerminationStrategy):
    # TODO etudiant : implementer la strategie de terminaison hybride
    PHRASES_ABANDON: list[str] = []  # Etape 1 : phrases de renoncement
    
    async def should_agent_terminate(self, agent, history: list[ChatMessageContent], cancellation_token=None) -> bool:
        # Etape 2 : verifier mot devine OU abandon detecte
        result = False  # TODO etudiant : remplacer par la logique
        return result

print("Exercice a completer : RenoncementTerminationStrategy")

Exercice a completer


## Discussion group configuration

In [7]:
# Bloc 6 - Configuration corrigée
chat = AgentGroupChat(
    agents=[pere_fouras, laurent_jalabert],
    termination_strategy=FortBoyardTerminationStrategy(
        agents=[laurent_jalabert],  # Définit explicitement les agents
        maximum_iterations=20       # Définit le nombre max d'itérations
    )
)


## Starting the game!

In [8]:
from semantic_kernel.contents import AuthorRole, ChatMessageContent

# Bloc 7 - Exécution du jeu
async def jouer_partie():
    logger.info("Depart du duel Pere Fouras vs Laurent Jalabert !")
    logger.info(f"Mot a deviner : {MOT_A_DEVINER.upper()}")
    
    # Message initial pour lancer la conversation
    await chat.add_chat_message(
        ChatMessageContent(role=AuthorRole.USER, content="Pere Fouras, donne-moi un indice !")
    )
    
    async for message in chat.invoke():
        role = message.role
        logger.info(f"[{role}] : {message.content}")
    
    logger.info("Partie terminee !")

await jouer_partie()

2026-05-13 02:23:40,910 [INFO] Depart du duel Pere Fouras vs Laurent Jalabert !


2026-05-13 02:23:40,911 [INFO] Mot a deviner : ANTICONSTITUTIONNELLEMENT


2026-05-13 02:23:40,912 [INFO] Adding `1` agent chat messages


2026-05-13 02:23:40,913 [INFO] Selected agent at index 0 (ID: 84766105-9a1b-4e7d-8d69-5927eb188f3b, name: Pere_Fouras)


2026-05-13 02:23:40,913 [INFO] Invoking agent Pere_Fouras


...

2026-05-13 02:23:54,911 [INFO] OpenAI usage: CompletionUsage(completion_tokens=19, prompt_tokens=1023, total_tokens=1042, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))


2026-05-13 02:23:54,912 [INFO] Evaluating termination criteria for 3bf0ebff-94d5-4c86-98f6-11a128686a02


2026-05-13 02:23:54,913 [INFO] Evaluated criteria for 3bf0ebff-94d5-4c86-98f6-11a128686a02, should terminate: True


2026-05-13 02:23:54,914 [INFO] [AuthorRole.ASSISTANT] : Ton indice pointe très clairement vers : **« anticonstitutionnellement »**.


2026-05-13 02:23:54,915 [INFO] Partie terminee !


### Exercise 3: Scoring system for the game

Currently, the game simply stops when the word is guessed or after 20 iterations. The objective is to add a **scoring system** that counts the number of questions asked by Laurent Jalabert and displays a final score.

**Objective**: write a function `evaluer_partie` that takes the message history and returns a dictionary containing the number of questions asked, whether the word was found, and a score (fewer questions = better score).

**Hints**:
- # Step 1: Count the messages where the role is that of Laurent Jalabert
- # Step 2: Check whether the secret word appears in the last message
- # Hint: the score can be `max(0, 100 - nb_questions * 10)`

In [1]:
def evaluer_partie(history_messages: list) -> dict:
    # TODO etudiant : implementer le systeme de score
    nb_questions = 0     # Etape 1 : compter les questions de Laurent
    mot_trouve = False   # Etape 2 : verifier la presence du mot
    
    # Indice : parcourir history_messages et compter les messages de Laurent
    score = None  # TODO etudiant : calculer le score
    return {"nb_questions": nb_questions, "mot_trouve": mot_trouve, "score": score}

print("Exercice a completer : systeme de score")

Exercice a completer
